In [ ]:
%matplotlib inline

In [ ]:
# Figure 1 data extraction (keep item ids for matched-subset filtering)
import json
import numpy as np

TRACK_M_JSON = "../runs/pilot_v1_2_20260717_1247_qwen3_4b.json"
TRACK_P_JSON = "../runs/redesign_R1_trackP_20260717_1535_qwen3_4b.json"

def extract_rows(path):
    with open(path) as f:
        rec = json.load(f)
    rows = []
    for it in rec["results"]:
        for pos in it["positions"]:
            rows.append((it["id"], float(pos["p_cond_t1"])))
    print(f"{path}: {len(rows)} positions")
    return rows

rows_m = extract_rows(TRACK_M_JSON)
rows_p = extract_rows(TRACK_P_JSON)
assert len(rows_m) == 186, f"Track M: {len(rows_m)} positions"
assert len(rows_p) == 96, f"Track P: {len(rows_p)} positions"

In [ ]:
# Figure 1: ECDF of distance from determinism, matched-subset comparison
import matplotlib as mpl
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
import matplotlib.pyplot as plt
import os

P_ITEMS = {"s1-01", "s1-02", "s1-04", "s1-06",
           "s1-11", "s1-12", "s2-01", "s2-02"}
CLIP = 1e-14

def to_d(rows, items=None):
    selected = [(i, p) for i, p in rows if items is None or i in items]
    ps = np.asarray([p for _, p in selected], dtype=float)
    assert np.all(np.isfinite(ps)), "non-finite probability found"
    assert np.all((0.0 <= ps) & (ps <= 1.0)), "probability out of [0, 1]"
    d = 1.0 - np.maximum(ps, 1.0 - ps)
    return np.clip(d, CLIP, None)

dM_all = to_d(rows_m)
dM_match = to_d(rows_m, P_ITEMS)
dP = to_d(rows_p)
assert len(dM_all) == 186, f"M all: {len(dM_all)}"
assert len(dM_match) == 96, f"M matched: {len(dM_match)}"
assert len(dP) == 96, f"P: {len(dP)}"

os.makedirs("figures", exist_ok=True)
fig, ax = plt.subplots(figsize=(6.4, 4.0))

def ecdf(ax, d, **kw):
    xs = np.sort(d)
    ys = np.arange(1, len(xs) + 1) / len(xs)
    ax.step(xs, ys, where="post", **kw)

ecdf(ax, dM_all, color="0.75", lw=1.0, ls="-",
     label="Track M, all 18 items (186 pos., reference)")
ecdf(ax, dM_match, color="tab:blue", lw=1.6, ls="-", marker="o",
     markevery=9, markersize=5, markerfacecolor="none",
     label="Track M, matched 8 items (96 pos.)")
ecdf(ax, dP, color="tab:orange", lw=1.6, ls="--", marker="^",
     markevery=9, markersize=5, markerfacecolor="none",
     label="Track P, 8 items (96 pos.)")

# saturation threshold and per-curve cumulative markers at d = 0.01
ax.axvline(1e-2, color="black", lw=0.9, ls=":",
           label="saturation threshold $d = 0.01$")
ax.text(0.8, 0.04,
        "share of positions with $d<0.01$:\n"
        f"M matched {100*(dM_match < 1e-2).mean():.1f}%,  "
        f"P {100*(dP < 1e-2).mean():.1f}%",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=10)

ax.set_xscale("log")
ax.set_xlim(CLIP, 0.55)
ax.set_ylim(0, 1.02)
ax.set_xlabel(r"$d = 1 - \max(p,\,1-p)$ (distance from determinism)",
              fontsize=14)
ax.set_ylabel("cumulative share of positions", fontsize=14)
ax.legend(fontsize=10, loc="upper left", frameon=False)
ax.tick_params(labelsize=11)
fig.tight_layout()
fig.savefig("figures/fig1_saturation.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 2: label effect under counterbalancing (A/B items, Track M)
# q_QQ under map-1 vs map-2, read from the pilot audit record.
import json, os
import numpy as np
import matplotlib.pyplot as plt

TRACK_M_JSON = "../runs/pilot_v1_2_20260717_1247_qwen3_4b.json"

with open(TRACK_M_JSON) as f:
    rec = json.load(f)

items, q_m1, q_m2 = [], [], []
for it in rec["results"]:
    if it.get("label_scheme") != "AB":
        continue
    items.append(it["id"])
    q_m1.append(float(it["mappings"]["map-1"]["point"]["q_QQ"]))
    q_m2.append(float(it["mappings"]["map-2"]["point"]["q_QQ"]))
q_m1, q_m2 = np.array(q_m1), np.array(q_m2)
print(f"{len(items)} A/B items extracted")

os.makedirs("figures", exist_ok=True)
fig, ax = plt.subplots(figsize=(6,6))
ax.plot([-1.05, 1.05], [-1.05, 1.05], color="0.6", lw=0.9, ls="--", zorder=1)
ax.scatter(q_m1, q_m2, s=45, marker="o", facecolors="none",
           edgecolors="tab:blue", linewidths=1.3, zorder=2)
for x, y, it in zip(q_m1, q_m2, items):
    if abs(x - y) > 0.3:
        ax.annotate(it, (x, y), textcoords="offset points",
                    xytext=(6, 4), fontsize=11)
ax.axhline(0, color="0.85", lw=0.7, zorder=0)
ax.axvline(0, color="0.85", lw=0.7, zorder=0)
ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)
ax.set_aspect("equal")
ax.set_xlabel(r"$q_{\mathrm{QQ}}$ under map-1", fontsize=14)
ax.set_ylabel(r"$q_{\mathrm{QQ}}$ under map-2", fontsize=14)
ax.tick_params(labelsize=11)
fig.tight_layout()
fig.savefig("figures/fig2_label_effect.pdf", bbox_inches="tight")
plt.show()